In [173]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,DateType, DoubleType

In [174]:
spark = (
    SparkSession.builder
    .appName("Ecommerce ETL Pipeline")
    .master("local[*]")
    .getOrCreate()
)

In [175]:
orders_raw_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("status", StringType(), True),
    StructField("total_amount", StringType(), True),
    StructField("discount_pct", StringType(), True),
])

order_items_raw_schema = StructType([
    StructField("item_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("unit_price", StringType(), True),
    StructField("category", StringType(), True),
])

customers_raw_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("signup_date", StringType(), True),
    StructField("country", StringType(), True),
    StructField("customer_tier", StringType(), True),
    StructField("email", StringType(), True),
])

returns_raw_schema = StructType([
    StructField("return_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("return_date", StringType(), True),
    StructField("reason", StringType(), True),
    StructField("refund_amount", StringType(), True),
])

In [176]:
orders_df = (
    spark.read
    .option("header", True)
    .schema(orders_raw_schema)
    .csv("data/data/orders.csv")
)

customers_df = (
    spark.read
    .option("header", True)
    .schema(customers_raw_schema)
    .csv("data/data/customers.csv")
)

order_items_df = (
    spark.read
    .option("header", True)
    .schema(order_items_raw_schema)
    .csv("data/data/order_items.csv")
)

returns_df = (
    spark.read
    .option("header", True)
    .schema(returns_raw_schema)
    .csv("data/data/returns.csv")
)

In [177]:
orders_df.show(5)

+--------+-----------+----------+--------+------------+------------+
|order_id|customer_id|order_date|  status|total_amount|discount_pct|
+--------+-----------+----------+--------+------------+------------+
|O0001478|     C00173|2023-03-29|refunded|     1239.08|        15.0|
|O0000893|     C00194|02/10/2023| shipped|      858.91|        25.0|
|O0000231|     C00446|2023-12-04| pending|      1920.8|        20.0|
|O0000466|     C00342|2024-05-27|refunded|     1432.04|        20.0|
|O0000931|     C00036|23/02/2024| shipped|         9.3|        null|
+--------+-----------+----------+--------+------------+------------+
only showing top 5 rows



In [178]:
order_items_df.show(5)

+----------+--------+----------+--------+----------+-----------+
|   item_id|order_id|product_id|quantity|unit_price|   category|
+----------+--------+----------+--------+----------+-----------+
|I000001866|O0000633|     P0289|       6|    364.01|     beauty|
|I000002153|O0000724|     P0460|       5|    395.26|home_garden|
|I000000203|O0000072|     P0249|       6|    327.38|     beauty|
|I000001503|O0000510|     P0341|       7|    496.84|       toys|
|I000002193|O0000743|     P0318|       1|    426.77|     sports|
+----------+--------+----------+--------+----------+-----------+
only showing top 5 rows



In [179]:
customers_df.show(5)

+-----------+-----------+--------+-------------+--------------------+
|customer_id|signup_date| country|customer_tier|               email|
+-----------+-----------+--------+-------------+--------------------+
|     C00247| 2018-09-10|   Ghana|       Silver|courtneyberger@ex...|
|     C00125| 2023-12-21|Ethiopia|         Gold|  abrown@example.com|
|     C00413| 2020-12-13|  Rwanda|     platinum|elizabeth18@examp...|
|     C00219| 2018-11-09|  Rwanda|       BRONZE| ugibson@example.org|
|     C00016| 09/05/2021| Senegal|         Gold|lynchgeorge@examp...|
+-----------+-----------+--------+-------------+--------------------+
only showing top 5 rows



In [180]:
returns_df.show(5)

+---------+--------+-----------+----------------+-------------+
|return_id|order_id|return_date|          reason|refund_amount|
+---------+--------+-----------+----------------+-------------+
|  R000215|O0000434| 2023-07-25|      wrong_item|      1141.82|
|  R000234|O0001069| 2024-05-21|       defective|       180.75|
|  R000268|O0001933| 2024-04-10|    arrived_late|        331.7|
|  R000181|O0000885| 2022-04-27|not_as_described|       129.94|
|  R000156|O0001473| 2023-08-11|       defective|       624.34|
+---------+--------+-----------+----------------+-------------+
only showing top 5 rows



In [181]:
def parse_mixed_date(column_name):
    return F.coalesce(
        F.to_date(F.col(column_name), "yyyy-MM-dd"),
        F.to_date(F.col(column_name), "dd/MM/yyyy")
    )

In [182]:
orders_casted_df = (
    orders_df
    .withColumn("order_date_casted", parse_mixed_date("order_date"))
    .withColumn("total_amount_casted", F.col("total_amount").cast(DoubleType()))
    .withColumn("discount_pct_casted", F.col("discount_pct").cast(DoubleType()))
)

rejected_orders_df = orders_casted_df.filter(
    (
        F.col("order_date").isNotNull() &
        F.col("order_date_casted").isNull()
    )
    |
    (
        F.col("total_amount").isNotNull() &
        F.col("total_amount_casted").isNull()
    )
    |
    (
        F.col("discount_pct").isNotNull() &
        F.col("discount_pct_casted").isNull()
    )
)

orders_clean_casted_df = (
    orders_casted_df
    .filter(
        ~(
            (
                F.col("order_date").isNotNull() &
                F.col("order_date_casted").isNull()
            )
            |
            (
                F.col("total_amount").isNotNull() &
                F.col("total_amount_casted").isNull()
            )
            |
            (
                F.col("discount_pct").isNotNull() &
                F.col("discount_pct_casted").isNull()
            )
        )
    )
    .select(
        "order_id",
        "customer_id",
        F.col("order_date_casted").alias("order_date"),
        "status",
        F.col("total_amount_casted").alias("total_amount"),
        F.col("discount_pct_casted").alias("discount_pct")
    )
)

In [183]:
order_items_casted_df = (
    order_items_df
    .withColumn("quantity_casted", F.col("quantity").cast(IntegerType()))
    .withColumn("unit_price_casted", F.col("unit_price").cast(DoubleType()))
)

rejected_order_items_df = order_items_casted_df.filter(
    (
        F.col("quantity").isNotNull() &
        F.col("quantity_casted").isNull()
    )
    |
    (
        F.col("unit_price").isNotNull() &
        F.col("unit_price_casted").isNull()
    )
)

order_items_clean_casted_df = (
    order_items_casted_df
    .filter(
        ~(
            (
                F.col("quantity").isNotNull() &
                F.col("quantity_casted").isNull()
            )
            |
            (
                F.col("unit_price").isNotNull() &
                F.col("unit_price_casted").isNull()
            )
        )
    )
    .select(
        "item_id",
        "order_id",
        "product_id",
        F.col("quantity_casted").alias("quantity"),
        F.col("unit_price_casted").alias("unit_price"),
        "category"
    )
)

In [184]:
customers_casted_df = (
    customers_df
    .withColumn("signup_date_casted", parse_mixed_date("signup_date"))
)

rejected_customers_df = customers_casted_df.filter(
    F.col("signup_date").isNotNull() &
    F.col("signup_date_casted").isNull()
)

customers_clean_casted_df = (
    customers_casted_df
    .filter(
        ~(
            F.col("signup_date").isNotNull() &
            F.col("signup_date_casted").isNull()
        )
    )
    .select(
        "customer_id",
        F.col("signup_date_casted").alias("signup_date"),
        "country",
        "customer_tier",
        "email"
    )
)

In [185]:
returns_casted_df = (
    returns_df
    .withColumn("return_date_casted", parse_mixed_date("return_date"))
    .withColumn("refund_amount_casted", F.col("refund_amount").cast(DoubleType()))
)

rejected_returns_df = returns_casted_df.filter(
    (
        F.col("return_date").isNotNull() &
        F.col("return_date_casted").isNull()
    )
    |
    (
        F.col("refund_amount").isNotNull() &
        F.col("refund_amount_casted").isNull()
    )
)

returns_clean_casted_df = (
    returns_casted_df
    .filter(
        ~(
            (
                F.col("return_date").isNotNull() &
                F.col("return_date_casted").isNull()
            )
            |
            (
                F.col("refund_amount").isNotNull() &
                F.col("refund_amount_casted").isNull()
            )
        )
    )
    .select(
        "return_id",
        "order_id",
        F.col("return_date_casted").alias("return_date"),
        "reason",
        F.col("refund_amount_casted").alias("refund_amount")
    )
)

In [186]:

orders_clean_df = (
    orders_clean_casted_df
    .dropDuplicates()
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("customer_id").isNotNull())
    .withColumn(
        "is_negative_amount",
        F.when(F.col("total_amount") < 0, True).otherwise(False)
    )
)

orders_clean_df.show(5)

+--------+-----------+----------+---------+------------+------------+------------------+
|order_id|customer_id|order_date|   status|total_amount|discount_pct|is_negative_amount|
+--------+-----------+----------+---------+------------+------------+------------------+
|O0000488|     C00485|2022-11-11|  shipped|     2452.55|         5.0|             false|
|O0001809|     C00474|2023-10-01|  shipped|     2116.34|        15.0|             false|
|O0000255|     C00364|2022-01-12|completed|     1386.02|        15.0|             false|
|O0000414|     C00054|2024-06-13|cancelled|     2127.47|         5.0|             false|
|O0000012|     C00287|2024-01-27| refunded|     1196.73|        15.0|             false|
+--------+-----------+----------+---------+------------+------------+------------------+
only showing top 5 rows



In [187]:

customers_clean_df = (
    customers_clean_casted_df
    .dropDuplicates()
    .filter(F.col("customer_id").isNotNull())
    .withColumn("customer_tier", F.lower(F.col("customer_tier")))
)

customers_clean_df.show(5)

+-----------+-----------+------------+-------------+--------------------+
|customer_id|signup_date|     country|customer_tier|               email|
+-----------+-----------+------------+-------------+--------------------+
|     C00130| 2020-01-16|       Kenya|     platinum|priceelizabeth@ex...|
|     C00248| 2018-12-04|South Africa|     platinum|parkerscott@examp...|
|     C00009| 2020-10-10|     Nigeria|       silver|  wdavis@example.net|
|     C00036| 2019-09-08|       Ghana|       bronze|nataliearroyo@exa...|
|     C00332| 2023-05-18|      Uganda|         gold|michelle52@exampl...|
+-----------+-----------+------------+-------------+--------------------+
only showing top 5 rows



In [188]:
order_items_clean_df = (
    order_items_clean_casted_df
    .dropDuplicates()
    .filter(F.col("item_id").isNotNull())
    .filter(F.col("order_id").isNotNull())
)

order_items_clean_df.show(5)

+----------+--------+----------+--------+----------+-----------+
|   item_id|order_id|product_id|quantity|unit_price|   category|
+----------+--------+----------+--------+----------+-----------+
|I000004891|O0001658|     P0201|       6|      46.1|electronics|
|I000003941|O0001348|     P0374|       5|    133.91|       toys|
|I000004690|O0001597|     P0427|       9|    437.65|       toys|
|I000003490|O0001181|     P0322|       4|    494.77|     beauty|
|I000002842|O0000955|     P0491|       4|     112.2|     beauty|
+----------+--------+----------+--------+----------+-----------+
only showing top 5 rows



In [189]:
returns_clean_df = (
    returns_clean_casted_df
    .dropDuplicates()
    .filter(F.col("return_id").isNotNull())
    .filter(F.col("order_id").isNotNull())
)
returns_clean_df.show(5)    

+---------+--------+-----------+------------+-------------+
|return_id|order_id|return_date|      reason|refund_amount|
+---------+--------+-----------+------------+-------------+
|  R000101|O0001028| 2022-04-01|changed_mind|       477.64|
|  R000172|O0000928| 2022-03-08|changed_mind|       669.45|
|  R000257|O0001091| 2022-07-29|  wrong_item|      1106.98|
|  R000253|O0000313| 2023-12-06|   defective|         null|
|  R000291|O0000936| 2022-11-12|  wrong_item|       450.11|
+---------+--------+-----------+------------+-------------+
only showing top 5 rows



In [190]:
orphaned_order_items_df = (
    order_items_clean_df
    .join(
        orders_clean_df.select("order_id").dropDuplicates(),
        on="order_id",
        how="left_anti"
    )
)
orphaned_order_items_df.show(5)

+-------------+----------+----------+--------+----------+--------+
|     order_id|   item_id|product_id|quantity|unit_price|category|
+-------------+----------+----------+--------+----------+--------+
|     O0001955|I000005822|     P0147|       4|    341.93|clothing|
|     O0001249|I000003667|     P0342|       6|     46.58|clothing|
|O_GHOST_07577|I000006180|     P0007|       5|    307.21|  beauty|
|O_GHOST_35488|I000006022|     P0292|       7|    115.62|  sports|
|O_GHOST_24035|I000006095|     P0055|       5|    296.21|  sports|
+-------------+----------+----------+--------+----------+--------+
only showing top 5 rows



In [191]:
orders_customers_df = (
    orders_clean_df
    .join(
        customers_clean_df,
        on="customer_id",
        how="inner"
    )
)
orders_customers_df.show(5)

+-----------+--------+----------+---------+------------+------------+------------------+-----------+------------+-------------+--------------------+
|customer_id|order_id|order_date|   status|total_amount|discount_pct|is_negative_amount|signup_date|     country|customer_tier|               email|
+-----------+--------+----------+---------+------------+------------+------------------+-----------+------------+-------------+--------------------+
|     C00485|O0000488|2022-11-11|  shipped|     2452.55|         5.0|             false| 2023-10-15|South Africa|       silver|oconnelltiffany@e...|
|     C00474|O0001809|2023-10-01|  shipped|     2116.34|        15.0|             false| 2019-12-20|South Africa|     platinum|shellyhendrix@exa...|
|     C00364|O0000255|2022-01-12|completed|     1386.02|        15.0|             false| 2021-11-07|    Tanzania|       bronze|angela18@example.org|
|     C00054|O0000414|2024-06-13|cancelled|     2127.47|         5.0|             false| 2022-09-15|South 

In [192]:
enriched_orders_df = (
    orders_customers_df
    .join(
        order_items_clean_df,
        on="order_id",
        how="left"
    )
    .withColumn(
        "net_amount",
        F.col("total_amount") * (1 - F.col("discount_pct") / 100)
    )
)
enriched_orders_df.show(5)

+--------+-----------+----------+---------+------------+------------+------------------+-----------+------------+-------------+--------------------+----------+----------+--------+----------+-----------+------------------+
|order_id|customer_id|order_date|   status|total_amount|discount_pct|is_negative_amount|signup_date|     country|customer_tier|               email|   item_id|product_id|quantity|unit_price|   category|        net_amount|
+--------+-----------+----------+---------+------------+------------+------------------+-----------+------------+-------------+--------------------+----------+----------+--------+----------+-----------+------------------+
|O0000488|     C00485|2022-11-11|  shipped|     2452.55|         5.0|             false| 2023-10-15|South Africa|       silver|oconnelltiffany@e...|I000001437|     P0420|       2|    203.97|     sports|         2329.9225|
|O0000488|     C00485|2022-11-11|  shipped|     2452.55|         5.0|             false| 2023-10-15|South Africa

In [193]:
returns_enriched_df = (
    returns_clean_df
    .join(
        enriched_orders_df,
        on="order_id",
        how="left"
    )
    .withColumn(
        "refund_exceeds_order",
        F.when(F.col("refund_amount") > F.col("net_amount"), True).otherwise(False)
    )
)
returns_clean_df.show(5)

+---------+--------+-----------+------------+-------------+
|return_id|order_id|return_date|      reason|refund_amount|
+---------+--------+-----------+------------+-------------+
|  R000101|O0001028| 2022-04-01|changed_mind|       477.64|
|  R000172|O0000928| 2022-03-08|changed_mind|       669.45|
|  R000257|O0001091| 2022-07-29|  wrong_item|      1106.98|
|  R000253|O0000313| 2023-12-06|   defective|         null|
|  R000291|O0000936| 2022-11-12|  wrong_item|       450.11|
+---------+--------+-----------+------------+-------------+
only showing top 5 rows



In [194]:
orders_per_tier_df = (
    enriched_orders_df
    .groupBy("customer_tier")
    .agg(F.countDistinct("order_id").alias("total_orders"))
)

returns_per_tier_df = (
    returns_enriched_df
    .groupBy("customer_tier")
    .agg(F.countDistinct("return_id").alias("total_returns"))
)

In [195]:
return_rate_by_tier_df = (
    orders_per_tier_df
    .join(
        returns_per_tier_df,
        on="customer_tier",
        how="left"
    )
    .fillna({"total_returns": 0})
    .withColumn(
        "return_rate",
        F.col("total_returns") / F.col("total_orders")
    )
)
return_rate_by_tier_df.show()

+-------------+------------+-------------+-------------------+
|customer_tier|total_orders|total_returns|        return_rate|
+-------------+------------+-------------+-------------------+
|       bronze|         543|           85|0.15653775322283608|
|       silver|         519|           89|0.17148362235067438|
|         gold|         375|           50|0.13333333333333333|
|     platinum|         477|           66|0.13836477987421383|
+-------------+------------+-------------+-------------------+



In [196]:
top_10_refund_customers_df = (
    returns_enriched_df
    .groupBy("customer_id", "email", "country", "customer_tier")
    .agg(F.sum("refund_amount").alias("total_refund_amount"))
    .orderBy(F.col("total_refund_amount").desc())
    .limit(10)
)
top_10_refund_customers_df.show()

+-----------+--------------------+------------+-------------+-------------------+
|customer_id|               email|     country|customer_tier|total_refund_amount|
+-----------+--------------------+------------+-------------+-------------------+
|     C00407|raymond43@example...|       Egypt|       silver|           13509.57|
|     C00306| kchavez@example.com|South Africa|       bronze|           11760.85|
|     C00296|uparrish@example.org|       Egypt|     platinum|            11314.7|
|       null|                null|        null|         null| 10334.120000000003|
|     C00492| nancy04@example.net|       Ghana|       bronze|           10062.65|
|     C00127|  wsmith@example.com|       Kenya|       bronze|            9709.55|
|     C00350|elizabethcalderon...|    Tanzania|       bronze|             9667.4|
|     C00298|ericolson@example...|    Ethiopia|       silver|            9545.01|
|     C00254|michelleramsey@ex...|       Ghana|     platinum|  9210.740000000002|
|     C00081|rod

In [197]:
print("Clean orders:", orders_clean_df.count())
print("Rejected orders:", rejected_orders_df.count())

print("Clean customers:", customers_clean_df.count())
print("Rejected customers:", rejected_customers_df.count())

print("Clean order items:", order_items_clean_df.count())
print("Rejected order items:", rejected_order_items_df.count())

print("Clean returns:", returns_clean_df.count())
print("Rejected returns:", rejected_returns_df.count())

print("Orphaned order items:", orphaned_order_items_df.count())
print("Enriched orders:", enriched_orders_df.count())

print("Customer rankings:", customers_ranked_df.count())
print("Rolling order count rows:", rolling_order_count_df.count())
print("Category revenue share rows:", category_revenue_share_df.count())

print("Returns enriched:", returns_enriched_df.count())
print("Return rate by category rows:", return_rate_by_category_df.count())
print("Return rate by tier rows:", return_rate_by_tier_df.count())
print("Top refund customers:", top_10_refund_customers_df.count())

Clean orders: 1914
Rejected orders: 0
Clean customers: 500
Rejected customers: 0
Clean order items: 6210
Rejected order items: 0
Clean returns: 318
Rejected returns: 0
Orphaned order items: 489
Enriched orders: 5721


NameError: name 'customers_ranked_df' is not defined